# Exercise 2: Code Generation with ReACT Prompting

[Open in Google Colab](https://colab.research.google.com/github/singhys0404/AAI2025/blob/2026fall/Prompt_Engineering/02_ReACT_Code_Generation.ipynb)

## Goal and setup
Generate, run, inspect, and improve Python code for response-time statistics.

Tools: Codex for code generation and revision; Python 3 standard library for actual execution; Colab for notebook execution; GitHub for sharing. No API key is required.

This is a recorded **AI-assisted development cycle**, not an autonomous runtime agent. The code was generated during authoring. Running this notebook executes both versions and reproduces the observed failure and successful fix. The initial function was executed and its empty-list failure was observed before the revision was authored.

## Full ReACT prompt

Act as a careful Python developer. Use a ReACT-style cycle with labeled stages: PLAN (a concise implementation rationale), ACT (generate code), RUN (execute it), OBSERVE (report actual results), and FIX (revise from observed failures and rerun). Do not claim execution without a real run.
Task: write response_stats(minutes, sla=30) for customer-support response times. Input is a list of finite, nonnegative Python integers/floats measured in minutes. Reject booleans, strings, negative numbers, NaN, infinity, and a non-list input with ValueError. Validate sla by the same numeric rules. Use Python 3 standard library only; no files, network, external packages, or user input. Return a dictionary containing count, average_minutes rounded to 2 decimals, and within_sla_percent rounded to 2 decimals. A response exactly at sla counts as within SLA. For an empty list, return count=0 and None for both statistics. Show the output for [12,25,40,18,55], retain a before/fix example, and test empty input, the SLA boundary, zero, and invalid inputs. Keep the final output clearly labeled.

## PLAN - concise implementation rationale
Compute the number of observations, their mean, and the fraction at or below the SLA threshold. Test normal input and empty input first, then use observed results to revise the implementation. The initial baseline below is intentionally retained as an incomplete first version.

In [1]:
def response_stats_v1(minutes, sla=30):
    return {"count": len(minutes),
            "average_minutes": round(sum(minutes) / len(minutes), 2),
            "within_sla_percent": round(100 * sum(x <= sla for x in minutes) / len(minutes), 2)}

print("ACT / RUN - version 1")
print("Normal input:", response_stats_v1([12, 25, 40, 18, 55]))
try:
    print("Empty input:", response_stats_v1([]))
except ZeroDivisionError as exc:
    observation = type(exc).__name__ + ": " + str(exc)
    print("OBSERVE - caught baseline failure:", observation)

ACT / RUN - version 1
Normal input: {'count': 5, 'average_minutes': 30.0, 'within_sla_percent': 60.0}
OBSERVE - caught baseline failure: ZeroDivisionError: division by zero


## OBSERVE → FIX
The first version gives the expected normal result but divides by zero on an empty list. It also lacks the required validation. The exception is caught so this documented baseline does not interrupt the notebook.

**Follow-up prompt**

Observed test result: response_stats_v1([12,25,40,18,55]) returns count=5, average_minutes=30.0, within_sla_percent=60.0, but response_stats_v1([]) raises ZeroDivisionError: division by zero. Revise the function to implement the original empty-input contract. Also add explicit list, numeric, finite, nonnegative, boolean, and SLA validation before computing the statistics. Preserve the normal-case result, and rerun tests including the exact SLA boundary and invalid inputs.

## ACT - revised code
Validate inputs before arithmetic and return `None` for undefined empty-list statistics. Use `<=` for the inclusive boundary. Boolean values need an explicit rejection because Python treats them as integers.

In [2]:
import math

def valid_minutes(value):
    """Accept finite, nonnegative int/float values, excluding bool."""
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        return False
    try:
        return math.isfinite(value) and value >= 0
    except OverflowError:
        return False

def response_stats(minutes, sla=30):
    """Return count, mean minutes, and percent at or below sla.

    Empty lists return None for undefined statistics. Invalid inputs raise
    ValueError; the input list is never modified.
    """
    if not isinstance(minutes, list):
        raise ValueError("minutes must be a list")
    if not valid_minutes(sla):
        raise ValueError("sla must be finite, nonnegative, and numeric (not bool)")
    if not all(valid_minutes(value) for value in minutes):
        raise ValueError("each time must be finite, nonnegative, and numeric (not bool)")
    count = len(minutes)
    if count == 0:
        return {"count": 0, "average_minutes": None, "within_sla_percent": None}
    # Divide before summing to avoid overflow from summing very large floats.
    average = math.fsum(value / count for value in minutes)
    within = sum(value <= sla for value in minutes)
    return {"count": count, "average_minutes": round(average, 2),
            "within_sla_percent": round(100 * within / count, 2)}

print("FIX / RERUN - version 2")
print("Normal input:", response_stats([12, 25, 40, 18, 55]))
print("Empty input:", response_stats([]))

FIX / RERUN - version 2
Normal input: {'count': 5, 'average_minutes': 30.0, 'within_sla_percent': 60.0}
Empty input: {'count': 0, 'average_minutes': None, 'within_sla_percent': None}


## RUN → OBSERVE - verify normal, boundary, and invalid inputs

In [3]:
checks = 0
def check_result(label, values, expected, sla=30):
    global checks
    assert response_stats(values, sla) == expected, label
    checks += 1
    print("PASS:", label)

check_result("normal", [12,25,40,18,55], {"count":5,"average_minutes":30.0,"within_sla_percent":60.0})
check_result("empty", [], {"count":0,"average_minutes":None,"within_sla_percent":None})
check_result("inclusive SLA boundary", [30,31], {"count":2,"average_minutes":30.5,"within_sla_percent":50.0})
check_result("zero time and zero SLA", [0], {"count":1,"average_minutes":0.0,"within_sla_percent":100.0}, sla=0)
check_result("rounding", [1,2,2], {"count":3,"average_minutes":1.67,"within_sla_percent":100.0})
invalid = [("negative",[-1],30), ("string",["5"],30), ("boolean",[True],30),
           ("NaN",[float("nan")],30), ("infinity",[float("inf")],30),
           ("not a list",(1,2),30), ("negative SLA",[5],-1),
           ("boolean SLA",[5],True), ("NaN SLA",[5],float("nan")),
           ("invalid SLA with empty list",[],"30")]
for label, values, sla in invalid:
    try:
        response_stats(values, sla)
    except ValueError:
        checks += 1
        print("PASS: rejected", label)
    else:
        raise AssertionError("Expected ValueError: " + label)
original = [12,25,40,18,55]
response_stats(original)
assert original == [12,25,40,18,55]
checks += 1
print("PASS: input remains unchanged")
print(f"{checks}/{checks} checks passed.")
print("FINAL:", response_stats(original))

PASS: normal
PASS: empty
PASS: inclusive SLA boundary
PASS: zero time and zero SLA
PASS: rounding
PASS: rejected negative
PASS: rejected string
PASS: rejected boolean
PASS: rejected NaN
PASS: rejected infinity
PASS: rejected not a list
PASS: rejected negative SLA
PASS: rejected boolean SLA
PASS: rejected NaN SLA
PASS: rejected invalid SLA with empty list
PASS: input remains unchanged
16/16 checks passed.
FINAL: {'count': 5, 'average_minutes': 30.0, 'within_sla_percent': 60.0}


## Takeaway
For the five sample response times, the mean is **30.0 minutes** and **60.0%** meet the 30-minute SLA. The repaired function handles empty input and rejects invalid data; all 16 checks pass. ReACT connects the revision to actual execution evidence rather than stopping after code generation.